# Finalizing Topic Pairs
<!-- structured-notebook -->
## Notebook Summary
Purpose: take the S-matrices produced by `01_cross_platform_matching.ipynb` and pick the stable, mutually-consistent topic pairs that will feed downstream cross-platform analyses.

Main steps:
- Load `S_news`, `S_social` matrices (coerce indices to int, move -1 outlier column to the end, assert row sums ≤ 1).
- Mutual top-1 pairing: keep pairs where A → B and B → A point to each other.
- Build a symmetric score `gmean = sqrt(S_ij · S_ji)`.
- Apply stable shortlist thresholds: `S_ij ≥ TA`, `S_ji ≥ TB`, topic sizes ≥ `MIN_TOPIC_SIZE`.
- Produce `pair_quality_mutual_top1.csv`, `stable_pairs_mutual_top1.csv`, `topk_pairs_bidir.csv`, `stable_pairs_readable.csv`.
- Sample documents per stable pair for human audit.


# Inputs / Outputs
<!-- io-table -->

| Role | File | Upstream / downstream |
|---|---|---|
| Input | `<DATA_ROOT>/topic_matchings/<pair>/BTM_S_*.csv` | Produced by `01_cross_platform_matching.ipynb` |
| Output | `<DATA_ROOT>/topic_matchings/<pair>/stable_pairs_readable.csv` | Consumed by `RQ2.ipynb`, `03_pair_visuals.ipynb` |
| Output | `<DATA_ROOT>/topic_matchings/<pair>/topk_pairs_bidir.csv` | Inspection artifact |


In [ ]:
from src.shared_reddit_telegram.config import (
    PUBMED_MODELS,
    REDDIT_MODELS,
)
from dataclasses import dataclass
from typing import Optional

@dataclass
class CorpusSpec:
    name: str
    model_path: str
    embedding_model_on_load: Optional[str] = None   # HF id or local dir; or None
    topics_csv: Optional[str] = None
    native_labels_path: Optional[str] = None        # npy/npz/csv/parquet
    docs_path: Optional[str] = None                 # pkl(list[str]) or csv
    docs_text_column: Optional[str] = None

@dataclass
class RunSpec:
    outdir: str
    s_A_vs_B_path: str          # rows=A topics, cols=B topics (+ -1)
    s_B_vs_A_path: str
    min_topic_size: int = 40
    thresh_A_to_B: float = 0.0  # TA
    thresh_B_to_A: float = 0.0  # TB
    topk: int = 3
    n_pairs_to_sample: int = 10
    n_docs_per_topic_sample: int = 10

In [ ]:
import re
from typing import Dict, Set, Tuple, List
import numpy as np

# Generic biomedical heads – keep *content* tokens
_GENERIC_HEADS = {
    "disease","disorder","syndrome","condition","therapy","treatment","study","trial",
    "deficiency","toxicity","infection","infections","injury","injuries","risk",
    "patients","patient","people","human","humans","model","models","mouse","mice",
    "aging","ageing","antiaging","anti-ageing"  # keep if you like; remove if too aggressive
}

# Quick alias table you can extend as you notice patterns
ALIASES = {
    "alzheimer's": "alzheimer",
    "alzheimer’s": "alzheimer",        # curly apostrophe
    "alzheimer disease": "alzheimer",
    "alzheimers": "alzheimer",
    "parkinson's": "parkinson",
    "parkinson’s": "parkinson",
    "type 2 diabetes": "t2d",
    "type ii diabetes": "t2d",
    "vit d": "vitamin d",
    "omega 3": "omega-3",
    "omega3": "omega-3",
    "intermittent fasting": "time-restricted feeding",
    "cr": "caloric restriction",
    "metformin hydrochloride": "metformin",
}

_token_re = re.compile(r"[A-Za-z0-9+-]+")

def _basic_norm(text: str) -> str:
    t = text.lower().strip()
    t = t.replace("’", "'")              # normalize quotes
    t = re.sub(r"'s\b", "", t)           # remove possessive (alzheimer's -> alzheimer)
    t = re.sub(r"[-_/]", " ", t)         # unify separators
    t = re.sub(r"\s+", " ", t).strip()
    # alias replacement on full phrase (cheap & effective)
    return ALIASES.get(t, t)

def _singularize(word: str) -> str:
    # super-light singularization (plural -> singular for common endings)
    if word.endswith("ies") and len(word) > 3:
        return word[:-3] + "y"
    if word.endswith("ses") and len(word) > 3:
        return word[:-2]  # e.g., "diseases" -> "disease"
    if word.endswith("s") and not word.endswith(("ss","us")) and len(word) > 3:
        return word[:-1]
    return word

def _stem(word: str) -> str:
    # optional ultra-light stemming for bio-ish words
    for suf in ("ing","ation","ations","ality","alities"):
        if word.endswith(suf) and len(word) > len(suf)+2:
            return word[: -len(suf)]
    return word

def normalize_phrase_to_content_tokens(phrase: str) -> Set[str]:
    t = _basic_norm(phrase)
    toks = [_singularize(w) for w in _token_re.findall(t)]
    toks = [_stem(w) for w in toks]
    toks = [w for w in toks if w and w not in _GENERIC_HEADS]
    return set(toks)

def normalize_wordlist_to_content_set(words: List[str]) -> Set[str]:
    out: Set[str] = set()
    for w in words:
        out |= normalize_phrase_to_content_tokens(w)
    return out

def containment_score(A: Set[str], B: Set[str]) -> float:
    if not A or not B:
        return 0.0
    inter = len(A & B)
    return inter / max(1, min(len(A), len(B)))

In [ ]:
import numpy as np
import pandas as pd
from typing import Dict, Set, Tuple, List
from scipy.optimize import linear_sum_assignment as hungarian

def _topic_word_weights(model, top_k: int = 20) -> Dict[int, Dict[str, float]]:
    out: Dict[int, Dict[str, float]] = {}
    for t, tuples in model.get_topics().items():
        if t == -1:
            continue
        out[int(t)] = {w: float(wt) for w, wt in (tuples[:top_k] or [])}
    return out

def _norm(d: Dict[str, float]) -> float:
    return float(np.sqrt(sum(v*v for v in d.values()))) if d else 1.0

def _cosine_kw(a: Dict[str, float], b: Dict[str, float]) -> float:
    if not a or not b:
        return 0.0
    if len(a) > len(b):
        a, b = b, a
    s = 0.0
    for w, va in a.items():
        vb = b.get(w)
        if vb is not None:
            s += va * vb
    na, nb = _norm(a), _norm(b)
    return float(s / (na * nb + 1e-12))

def get_topic_doc_counts_from_csv(topics_csv):
    try:
        df = pd.read_csv(topics_csv)
        # Try to find the topic id column (could be 'Topic' or similar)
        topic_col = 'Topic' if 'Topic' in df.columns else df.columns[0]
        count_col = 'Count' if 'Count' in df.columns else None
        if count_col is None:
            return {}
        return dict(zip(df[topic_col], df[count_col]))
    except Exception:
        return {}

def keyword_pairs(
    model_a, model_b,
    top_k_words: int = 25,
    min_cos: float = 0.22,
    min_jacc: float = 0.06,
    min_contain: float = 0.30,   # NEW: require decent containment
    one_to_one: bool = True,
    score_weights: Tuple[float,float,float] = (0.5, 0.2, 0.3)  # (cos, jaccard, containment)
) -> pd.DataFrame:
    """
    Pair topics using only keywords enriched with robust normalization:
    - cosine on c-TF-IDF weights (original words)
    - Jaccard on normalized content tokens
    - Containment on normalized content tokens (handles 'alzheimer' vs 'alzheimer disease')
    """
    WA = _topic_word_weights(model_a, top_k=top_k_words)
    WB = _topic_word_weights(model_b, top_k=top_k_words)
    idsA = sorted(WA.keys()); idsB = sorted(WB.keys())

    info_a = model_a.get_topic_info().set_index("Topic")
    info_b = model_b.get_topic_info().set_index("Topic")
    name_a = info_a["Name"] if "Name" in info_a else pd.Series(dtype=str)
    name_b = info_b["Name"] if "Name" in info_b else pd.Series(dtype=str)

    # Precompute normalized token sets for every topic (from the *strings* of top words)
    A_tokens: Dict[int, Set[str]] = {}
    B_tokens: Dict[int, Set[str]] = {}
    for i in idsA:
        A_tokens[i] = normalize_wordlist_to_content_set(list(WA[i].keys()))
    for j in idsB:
        B_tokens[j] = normalize_wordlist_to_content_set(list(WB[j].keys()))

    # Get document counts per topic for model_a and model_b
    def get_topic_doc_counts(model, corpus_spec=None):
        if corpus_spec and corpus_spec.topics_csv:
            return get_topic_doc_counts_from_csv(corpus_spec.topics_csv)
        try:
            doc_info = model.get_document_info()
            topic_counts = doc_info['Topic'].value_counts().to_dict()
        except Exception:
            topic_counts = {}
        return topic_counts

    # Find corpus_spec for model_a and model_b if available
    corpus_a = None
    corpus_b = None
    for spec in [youtube, news, pubmed, pubmed_allenai, pubmed_simple, reddit, test]:
        if hasattr(model_a, 'model_path') and getattr(model_a, 'model_path', None) == spec.model_path:
            corpus_a = spec
        if hasattr(model_b, 'model_path') and getattr(model_b, 'model_path', None) == spec.model_path:
            corpus_b = spec

    A_doc_counts = get_topic_doc_counts(model_a, corpus_a)
    B_doc_counts = get_topic_doc_counts(model_b, corpus_b)

    cands = []
    w_cos, w_jac, w_con = score_weights
    for i in idsA:
        for j in idsB:
            c = _cosine_kw(WA[i], WB[j])
            if c < min_cos:
                continue
            # token-level signals
            At, Bt = A_tokens[i], B_tokens[j]
            if not At or not Bt:
                continue
            inter = len(At & Bt)
            if inter == 0:
                continue
            jac = inter / len(At | Bt)
            con = containment_score(At, Bt)
            if jac < min_jacc or con < min_contain:
                continue
            score = w_cos * c + w_jac * jac + w_con * con
            cands.append((i, j, c, jac, con, inter, score))

    if not cands:
        return pd.DataFrame(columns=["A","B","cos_kw","jacc","contain","overlap_terms","A_name","B_name","common_terms","A_doc_count","B_doc_count"])

    if not one_to_one:
        df = pd.DataFrame(cands, columns=["A","B","cos_kw","jacc","contain","overlap_n","score"])
        df = df.sort_values(["A","score"], ascending=[True, False])
        df = df.groupby("A", as_index=False).first()
    else:
        A_list = sorted({i for i, *_ in cands})
        B_list = sorted({j for _, j, *_ in cands})
        a_idx = {i:k for k,i in enumerate(A_list)}
        b_idx = {j:k for k,j in enumerate(B_list)}
        cost = np.full((len(A_list), len(B_list)), 1e3, dtype=np.float64)
        best = {}
        for i,j,c,jac,con,ov,score in cands:
            if (i,j) not in best or score > best[(i,j)][-1]:
                best[(i,j)] = (c,jac,con,ov,score)
        for (i,j),(c,jac,con,ov,score) in best.items():
            cost[a_idx[i], b_idx[j]] = -score
        rows, cols = hungarian(cost)
        chosen = []
        for r, cc in zip(rows, cols):
            if cost[r, cc] >= 1e3:
                continue
            i, j = A_list[r], B_list[cc]
            c, jac, con, ov, score = best[(i,j)]
            chosen.append((i,j,c,jac,con,ov))
        df = pd.DataFrame(chosen, columns=["A","B","cos_kw","jacc","contain","overlap_n"])

    # decorate with names + explicit common terms
    def terms(i, j, k=12):
        Aset, Bset = A_tokens[int(i)], B_tokens[int(j)]
        return ", ".join(sorted(Aset & Bset)[:k])

    df["A_name"] = df["A"].map(name_a)
    df["B_name"] = df["B"].map(name_b)
    df["common_terms"] = df.apply(lambda r: terms(r["A"], r["B"]), axis=1)
    # Add document counts
    df["A_doc_count"] = df["A"].map(lambda x: A_doc_counts.get(x, 0))
    df["B_doc_count"] = df["B"].map(lambda x: B_doc_counts.get(x, 0))
    df = df.sort_values(["contain","jacc","cos_kw"], ascending=False)
    return df

In [ ]:
def _char_ngrams(s: str, n: int = 3) -> set[str]:
    s = re.sub(r"\s+", " ", s.lower().strip())
    s = s.replace("’", "'")
    s = re.sub(r"[^a-z0-9 +\-]", "", s)  # keep alnum, plus, hyphen, space
    if len(s) < n:
        return {s} if s else set()
    return {s[i:i+n] for i in range(len(s)-n+1)}

def _ngram_jacc(words_a: list[str], words_b: list[str], n: int = 3) -> float:
    # character n-gram overlap over joined phrases of top words
    A = _char_ngrams(" ".join(words_a), n)
    B = _char_ngrams(" ".join(words_b), n)
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

def keyword_pairs_second_variant(
    corpus_a: CorpusSpec, corpus_b: CorpusSpec,
    top_k_words: int = 35,
    # base thresholds
    min_cos: float = 0.18,
    min_jacc: float = 0.04,
    min_contain: float = 0.20,
    # backoff/adaptive controls
    target_ratio: float = 0.40,   # aim for ~40% of min(#topics_A,#topics_B)
    max_backoff_steps: int = 3,
    one_to_one: bool = True,
    score_weights: tuple[float,float,float] = (0.5, 0.2, 0.3),  # (cos, jacc, contain)
    use_ngram_backoff: bool = True,
) -> pd.DataFrame:
    """
    Robust keyword pairing with adaptive threshold backoff and OR-of-two gating.
    topics_a / topics_b: DataFrames containing at least ['Topic','Count'] for mapping counts.
    """

    model_a = BERTopic.load(corpus_a.model_path, embedding_model=corpus_a.embedding_model_on_load)
    model_b = BERTopic.load(corpus_b.model_path, embedding_model=corpus_b.embedding_model_on_load)
    topics_a = pd.read_csv(corpus_a.topics_csv)
    topics_b = pd.read_csv(corpus_b.topics_csv)
    corpus_name_a = corpus_a.name
    corpus_name_b = corpus_b.name

    # 1) Pull top words + names
    WA = _topic_word_weights(model_a, top_k=top_k_words)  # {topic -> {word: weight}}
    WB = _topic_word_weights(model_b, top_k=top_k_words)
    idsA = sorted(WA.keys()); idsB = sorted(WB.keys())

    info_a = model_a.get_topic_info().set_index("Topic")
    info_b = model_b.get_topic_info().set_index("Topic")
    name_a = info_a["Name"] if "Name" in info_a else pd.Series(dtype=str)
    name_b = info_b["Name"] if "Name" in info_b else pd.Series(dtype=str)

    # Build count maps directly from provided topic DataFrames
    def _build_count_map(df: pd.DataFrame):
        # Try to locate columns flexibly
        if df is None or df.empty:
            return {}
        cols = {c.lower(): c for c in df.columns}
        topic_col = cols.get('topic') or list(df.columns)[0]
        count_col = cols.get('count')
        if count_col is None:
            return {}
        sub = df[[topic_col, count_col]].dropna()
        return {int(t): int(c) for t, c in zip(sub[topic_col], sub[count_col]) if pd.notnull(t)}

    A_doc_counts = _build_count_map(topics_a)
    B_doc_counts = _build_count_map(topics_b)

    # 2) Precompute normalized token sets (content tokens) + raw word lists for n-gram fallback
    A_tokens = {i: normalize_wordlist_to_content_set(list(WA[i].keys())) for i in idsA}
    B_tokens = {j: normalize_wordlist_to_content_set(list(WB[j].keys())) for j in idsB}
    A_words  = {i: list(WA[i].keys()) for i in idsA}
    B_words  = {j: list(WB[j].keys()) for j in idsB}

    def build_candidates(cos_thr, jacc_thr, contain_thr, allow_ngram=False):
        w_cos, w_jac, w_con = score_weights
        cands = []
        for i in idsA:
            for j in idsB:
                # cosine on weighted keywords
                c = _cosine_kw(WA[i], WB[j])

                # token-level
                At, Bt = A_tokens[i], B_tokens[j]
                inter = len(At & Bt)
                jac = (inter / len(At | Bt)) if (At and Bt) else 0.0
                con = containment_score(At, Bt) if (At and Bt) else 0.0

                # optional char n-gram Jaccard
                ng = _ngram_jacc(A_words[i], B_words[j], n=3) if allow_ngram else 0.0

                # OR-of-two gating:
                # at least two of the three (cos, jacc, contain) must meet thresholds;
                # if n-gram is enabled, allow it to substitute for a weak jacc/contain.
                passes = 0
                passes += int(c >= cos_thr)
                passes += int(jac >= jacc_thr)
                passes += int(con >= contain_thr)
                if allow_ngram and ng >= max(0.20, jacc_thr):  # modest bar for n-gram
                    # treat as a bonus pass if jacc or contain is weak
                    passes = max(passes, 2)

                if passes < 2:
                    continue

                score = w_cos * c + w_jac * jac + w_con * con + (0.05 * ng if allow_ngram else 0.0)
                cands.append((i, j, c, jac, con, ng, inter, score))
        return cands

    # 3) Adaptive backoff loop
    target = int(round(min(len(idsA), len(idsB)) * target_ratio))
    cos_thr, jacc_thr, contain_thr = min_cos, min_jacc, min_contain
    allow_ngram = False
    chosen_df = None

    for step in range(max_backoff_steps + 1):
        cands = build_candidates(cos_thr, jacc_thr, contain_thr, allow_ngram)

        if not cands:
            # backoff more aggressively next round
            cos_thr = max(0.0, cos_thr - 0.03)
            jacc_thr = max(0.0, jacc_thr - 0.02)
            contain_thr = max(0.0, contain_thr - 0.05)
            allow_ngram = use_ngram_backoff or allow_ngram
            continue

        # one-to-one or per-A best
        if not one_to_one:
            df = pd.DataFrame(cands, columns=[f"{corpus_name_a}_topic_num",f"{corpus_name_b}_topic_num","cos_kw","jacc","contain","ngram3","overlap_n","score"])
            df = df.sort_values([f"{corpus_name_a}_topic_num","score"], ascending=[True, False]).groupby(f"{corpus_name_a}_topic_num", as_index=False).first()
        else:
            A_list = sorted({i for i, *_ in cands})
            B_list = sorted({j for _, j, *_ in cands})
            a_idx = {i:k for k,i in enumerate(A_list)}
            b_idx = {j:k for k,j in enumerate(B_list)}
            cost = np.full((len(A_list), len(B_list)), 1e3, dtype=np.float64)
            best = {}
            for i,j,cosv,jac,con,ng,ov,score in cands:
                if (i,j) not in best or score > best[(i,j)][-1]:
                    best[(i,j)] = (cosv,jac,con,ng,ov,score)
            for (i,j),(cosv,jac,con,ng,ov,score) in best.items():
                cost[a_idx[i], b_idx[j]] = -score
            rows, cols = hungarian(cost)
            chosen = []
            for r, cc in zip(rows, cols):
                if cost[r, cc] >= 1e3:
                    continue
                i, j = A_list[r], B_list[cc]
                cosv,jac,con,ng,ov,score = best[(i,j)]
                chosen.append((i,j,cosv,jac,con,ng,ov))
            df = pd.DataFrame(chosen, columns=[f"{corpus_name_a}_topic_num",f"{corpus_name_b}_topic_num","cos_kw","jacc","contain","ngram3","overlap_n"])

        # stop if we hit target or exhausted backoff
        chosen_df = df
        if len(df) >= max(1, target) or step == max_backoff_steps:
            break

        # relax thresholds a bit for next loop
        cos_thr = max(0.0, cos_thr - 0.02)
        jacc_thr = max(0.0, jacc_thr - 0.01)
        contain_thr = max(0.0, contain_thr - 0.03)
        allow_ngram = use_ngram_backoff or allow_ngram

    if chosen_df is None or chosen_df.empty:
        return pd.DataFrame(columns=[f"{corpus_name_a}_topic_num",f"{corpus_name_b}_topic_num","cos_kw","jacc","contain","ngram3","overlap_terms",f"{corpus_name_a}",f"{corpus_name_b}","common_terms",f"{corpus_name_a}_doc_count",f"{corpus_name_b}_doc_count"])

    # 4) decorate with names and explicit common terms
    def terms(i, j, k=12):
        Aset = normalize_wordlist_to_content_set(list(WA[int(i)].keys()))
        Bset = normalize_wordlist_to_content_set(list(WB[int(j)].keys()))
        return ", ".join(sorted(Aset & Bset)[:k])

    chosen_df[f"{corpus_name_a}"] = chosen_df[f"{corpus_name_a}_topic_num"].map(name_a)
    chosen_df[f"{corpus_name_b}"] = chosen_df[f"{corpus_name_b}_topic_num"].map(name_b)
    chosen_df["common_terms"] = chosen_df.apply(lambda r: terms(r[f"{corpus_name_a}_topic_num"], r[f"{corpus_name_b}_topic_num"]), axis=1)
    # Map counts from provided DataFrames
    chosen_df[f"{corpus_name_a}_doc_count"] = chosen_df[f"{corpus_name_a}_topic_num"].map(lambda x: A_doc_counts.get(int(x), 0))
    chosen_df[f"{corpus_name_b}_doc_count"] = chosen_df[f"{corpus_name_b}_topic_num"].map(lambda x: B_doc_counts.get(int(x), 0))
    chosen_df = chosen_df.sort_values(["contain","jacc","cos_kw"], ascending=False)
    return chosen_df

In [ ]:
youtube = CorpusSpec(
    name="youtube",
    model_path=REDDIT_MODELS / "round_11" / " 4/mpnet_topic_model_no_umap",
    topics_csv=REDDIT_MODELS / "round_11" / " 4/mpnet_topics.csv",
    native_labels_path="",
    docs_path="",
)

news = CorpusSpec(
    name="news",
    model_path=REDDIT_MODELS / "round_11" / "topic_model_noUmap",
    topics_csv=REDDIT_MODELS / "round_11" / "topic_info.csv",
    native_labels_path="",
    docs_path="",
)

pubmed = CorpusSpec(
    name="pubmed",
    model_path=PUBMED_MODELS / "10000_docs_all-mpnet_umap_hdbscan/BERTopic_model/bertopic_model",
    topics_csv=PUBMED_MODELS / "10000_docs_all-mpnet_umap_hdbscan/BERTopic_model/topics.csv",
    native_labels_path="doc-topicId-map.parquet",
    docs_path=PUBMED_MODELS / "10000_docs_all-mpnet_umap_hdbscan/BERTopic_model/preprocessed_docs.pkl",
)

pubmed_allenai = CorpusSpec(
    name="pubmed_allenai",
    model_path=PUBMED_MODELS / "10000_docs_allenai_umap_hdbscan/BERTopic_model/bertopic_model",
    topics_csv=PUBMED_MODELS / "10000_docs_allenai_umap_hdbscan/BERTopic_model/topics.csv",
    native_labels_path=PUBMED_MODELS / "10000_docs_allenai_umap_hdbscan/BERTopic_model/doc-topicId-map.parquet",
    docs_path=PUBMED_MODELS / "10000_docs_allenai_umap_hdbscan/BERTopic_model/preprocessed_docs.pkl",
)

pubmed_simple = CorpusSpec(
    name="pubmed_simple",
    model_path=PUBMED_MODELS / "10000_docs_all-mpnet-2/BERTopic_model/bertopic_model",
    topics_csv=PUBMED_MODELS / "10000_docs_all-mpnet-2/BERTopic_model/TopicSummary_PubMed.csv",
    native_labels_path=PUBMED_MODELS / "10000_docs_all-mpnet-2/BERTopic_model/doc-topicId-map.parquet",
    docs_path=PUBMED_MODELS / "10000_docs_all-mpnet-2/BERTopic_model/preprocessed_docs.pkl",
)

reddit = CorpusSpec(
    name="reddit",
    model_path="../topic_modelling_v2/round_10/bertopic_no_embed",
    topics_csv="../topic_modelling_v2/round_10/topics.csv",
    native_labels_path="../topic_modelling_v2/round_10/train_topics_unique.npy",
    docs_path="../topic_modelling_v2/round_4/unique_docs.pkl",
)

test = CorpusSpec(
    name="test",
    model_path="../topic_modelling_v2/round_10/bertopic_no_embed",
    topics_csv="",
    native_labels_path="",
    docs_path="",
)

R = RunSpec(
    outdir="reddit-youtube/round_4",
    s_A_vs_B_path="pubmed_simple-news/round_1/BTM_S_pubmed2_vs_youtube.csv",
    s_B_vs_A_path="pubmed_simple-news/round_1/BTM_S_youtube_vs_pubmed2.csv",
    min_topic_size=0, thresh_A_to_B=0.0, thresh_B_to_A=0.0, topk=3,
    n_pairs_to_sample=10, n_docs_per_topic_sample=10,
)

In [ ]:
from bertopic import BERTopic
import os


pairs_kw = keyword_pairs_second_variant(
    reddit, youtube,
    top_k_words=35,    # try 20–30
    min_cos=0.18,      # raise to 0.30–0.35 for stricter matches
    min_jacc=0.05,     # raise to 0.10–0.15 if you want clear term overlap
    min_contain=0.25,
    one_to_one=True    # set False for best-B per A (many-to-many-ish)
)
os.makedirs(os.path.join(R.outdir, "topic-pairs"), exist_ok=True)
pairs_kw.to_csv(os.path.join(R.outdir, "topic-pairs/keyword_pairs_2.csv"), index=False)
print(pairs_kw.head(10))

In [ ]:
csv_path = "pubmed_simple-news/round_1/topic-pairs/keyword_pairs_2.csv"
df = pd.read_csv(csv_path)
df_filtered = df[(df["A"] >= 20) & (df["B"] >= 30)]

df_filtered.to_csv("pubmed_simple-news/round_1/topic-pairs/keyword_pairs_2_filtered.csv", index=False)

Graph building from pairings

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Cross-platform topic cluster discovery + graph building (MERGED)

What you get:
- One pass CSV reader with consistent parsing across all steps
- Summarize connected components into multi-platform clusters
- Build a NetworkX graph with averaged scores + term aggregation
- Export nodes/edges CSV
- PyVis interactive HTML + optional legend; static PNG/Plotly fallbacks
- Minimal CLI for end-to-end execution

Author: merged for TDK-2025 project
"""

from __future__ import annotations

import os
import json
import math
import argparse
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Iterable, Set
from collections import defaultdict, Counter

import pandas as pd

try:
    import networkx as nx  # type: ignore
except Exception:
    nx = None  # we guard uses later

# -----------------------------
# Shared datatypes
# -----------------------------
ScoreDict = Dict[str, float]
Node = Tuple[str, int]  # (platform, topic_id)
EdgeKey = Tuple[Node, Node]


# -----------------------------
# CSV loading (single source of truth)
# -----------------------------
@dataclass
class PairRow:
    plat_a: str
    topic_a: int
    plat_b: str
    topic_b: int
    scores: ScoreDict
    common_terms: List[str]
    # Optional metadata for labeling and doc counts (if present)
    label_a: Optional[str] = None
    label_b: Optional[str] = None
    docs_a: Optional[int] = None
    docs_b: Optional[int] = None
    source_file: Optional[str] = None

# --- Aggregate common terms per undirected edge (union & intersection) ---
def _aggregate_edge_terms(pair_rows: List[PairRow]) -> dict[EdgeKey, dict]:
    """
    Build { edge_key: { 'union': set[str], 'intersection': set[str] } } across all CSV rows.
    Edge key is undirected: tuple(sorted([node_a, node_b])).
    """
    agg: dict[EdgeKey, dict] = {}
    def ek(a: Node, b: Node) -> EdgeKey:
        return tuple(sorted([a, b]))

    for pr in pair_rows:
        na, nb = (pr.plat_a, pr.topic_a), (pr.plat_b, pr.topic_b)
        key = ek(na, nb)
        terms = set(pr.common_terms or [])
        if key not in agg:
            # first occurrence initializes both union and intersection
            agg[key] = {"union": set(terms), "intersection": set(terms)}
        else:
            agg[key]["union"].update(terms)
            agg[key]["intersection"].intersection_update(terms)
    return agg

def _topic_cols(df: pd.DataFrame) -> Optional[Tuple[str, str]]:
    cols = [c for c in df.columns if c.endswith("_topic_num")]
    if len(cols) != 2:
        return None
    return cols[0], cols[1]


def _parse_platform(colname: str) -> str:
    # strip "_topic_num"
    return colname[:-10]

def filter_components_by_platforms(
    G: "nx.Graph",
    min_platforms: int = 3,
    require_all_platforms: bool = False,
    all_platforms: Optional[set[str]] = None,
    relabel_cluster_ids: bool = True,
) -> "nx.Graph":
    """Keep only components that meet platform-count criteria, then (optionally) relabel cluster_id to 1..K."""
    import networkx as nx  # local import to avoid top-level dependency issues

    if G.number_of_nodes() == 0:
        return G

    keep_nodes = set()
    for comp in nx.connected_components(G):
        plats = {p for (p, _) in comp}
        if len(plats) < min_platforms:
            continue
        if require_all_platforms and all_platforms and plats != set(all_platforms):
            continue
        keep_nodes |= comp

    H = G.subgraph(keep_nodes).copy()

    if relabel_cluster_ids and H.number_of_nodes() > 0:
        # Recompute components and assign dense cluster ids
        for cid, comp in enumerate(sorted(nx.connected_components(H), key=len, reverse=True), start=1):
            for n in comp:
                H.nodes[n]["cluster_id"] = cid

    return H


def load_pair_csvs(
    paths: List[str],
    drop_low_scores: bool = False,
    min_cos_kw: float = 0.0,
    min_jacc: float = 0.0,
    min_contain: float = 0.0,
    stop_terms: Optional[Set[str]] = None
) -> List[PairRow]:
    """Load all given pair CSVs into unified PairRow records."""
    stop_terms = {t.lower() for t in (stop_terms or set())}
    rows: List[PairRow] = []
    score_fields = ("cos_kw", "jacc", "contain", "ngram3")

    for path in paths:
        if not os.path.exists(path):
            continue
        try:
            df = pd.read_csv(path)
        except Exception:
            continue
        if df.empty:
            continue

        tc = _topic_cols(df)
        if tc is None:
            continue
        a_col, b_col = tc
        plat_a, plat_b = _parse_platform(a_col), _parse_platform(b_col)

        # optional columns
        name_a = plat_a if plat_a in df.columns else None
        name_b = plat_b if plat_b in df.columns else None
        doc_a = f"{plat_a}_doc_count" if f"{plat_a}_doc_count" in df.columns else None
        doc_b = f"{plat_b}_doc_count" if f"{plat_b}_doc_count" in df.columns else None

        for _, r in df.iterrows():
            if pd.isna(r[a_col]) or pd.isna(r[b_col]):
                continue
            try:
                ta = int(r[a_col]); tb = int(r[b_col])
            except Exception:
                continue

            # score gating
            if drop_low_scores:
                if "cos_kw" in df.columns and float(r.get("cos_kw", 0.0)) < min_cos_kw:
                    continue
                if "jacc" in df.columns and float(r.get("jacc", 0.0)) < min_jacc:
                    continue
                if "contain" in df.columns and float(r.get("contain", 0.0)) < min_contain:
                    continue

            scores: ScoreDict = {}
            for sf in score_fields:
                if sf in df.columns and not pd.isna(r.get(sf)):
                    scores[sf] = float(r[sf])

            # terms
            ct_list: List[str] = []
            if "common_terms" in df.columns and isinstance(r.get("common_terms"), str):
                terms = [t.strip().lower() for t in r["common_terms"].split(",") if t.strip()]
                ct_list = [t for t in terms if t not in stop_terms]

            label_a = r.get(name_a) if name_a else None
            label_b = r.get(name_b) if name_b else None
            docs_a = int(r[doc_a]) if doc_a and not pd.isna(r.get(doc_a)) else None
            docs_b = int(r[doc_b]) if doc_b and not pd.isna(r.get(doc_b)) else None

            rows.append(PairRow(
                plat_a=plat_a, topic_a=ta, plat_b=plat_b, topic_b=tb,
                scores=scores, common_terms=ct_list,
                label_a=label_a if pd.notna(label_a) else None,
                label_b=label_b if pd.notna(label_b) else None,
                docs_a=docs_a, docs_b=docs_b,
                source_file=os.path.basename(path)
            ))
    return rows


# -----------------------------
# Cluster discovery (union-find)
# -----------------------------
class _UF:
    def __init__(self):
        self.parent: Dict[Node, Node] = {}
        self.rank: Dict[Node, int] = {}

    def find(self, x: Node) -> Node:
        if self.parent.get(x, x) != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent.get(x, x)

    def union(self, a: Node, b: Node) -> None:
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank.get(ra, 0) < self.rank.get(rb, 0):
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank.get(ra, 0) == self.rank.get(rb, 0):
            self.rank[ra] = self.rank.get(ra, 0) + 1

# --- Single-source component discovery with STABLE cluster IDs ---
def _discover_components_with_ids(
    pair_rows: List[PairRow],
    min_platforms: int = 3,
    require_all_platforms: bool = False,
    require_term_intersection: bool = False,     # <-- NEW
) -> tuple[dict[Node, int], list[set[Node]]]:
    """
    Build union-find components once, optionally ignoring edges whose
    aggregated common-term intersection is empty.
    """
    uf = _UF()
    all_platforms: set[str] = set()

    # Precompute edge term intersections (used if flag is on)
    edge_terms = _aggregate_edge_terms(pair_rows) if require_term_intersection else None

    def allowed_edge(na: Node, nb: Node) -> bool:
        if not require_term_intersection:
            return True
        key = tuple(sorted([na, nb]))
        et = edge_terms.get(key)
        return bool(et and et["intersection"])  # must exist and non-empty

    # Build UF with optional edge gating
    for pr in pair_rows:
        all_platforms.update([pr.plat_a, pr.plat_b])
        na, nb = (pr.plat_a, pr.topic_a), (pr.plat_b, pr.topic_b)
        if not allowed_edge(na, nb):
            continue
        uf.parent.setdefault(na, na)
        uf.parent.setdefault(nb, nb)
        uf.union(na, nb)

    # Group by root
    root_to_nodes: dict[Node, set[Node]] = defaultdict(set)
    for node in uf.parent.keys():
        root = uf.find(node)
        root_to_nodes[root].add(node)

    # Filter components by platform rule
    comps: list[set[Node]] = []
    for nodes in root_to_nodes.values():
        plats = {p for (p, _) in nodes}
        if len(plats) < min_platforms:
            continue
        if require_all_platforms and plats != all_platforms:
            continue
        comps.append(nodes)

    if not comps:
        return {}, []

    # Deterministic cluster IDs (stable order)
    def comp_sig(nodes: set[Node]) -> tuple:
        return min((f"{p}:{t}" for (p, t) in nodes))
    comps_sorted = sorted(comps, key=comp_sig)

    node2cid: dict[Node, int] = {}
    for cid, comp in enumerate(comps_sorted, start=1):
        for n in comp:
            node2cid[n] = cid
    return node2cid, comps_sorted

def summarize_cross_platform_pairs(
    pair_rows: List[PairRow],
    min_platforms: int = 3,
    require_all_platforms: bool = False,
    save_path: Optional[str] = None,
    common_terms_cap: int = 30,
    require_term_intersection: bool = False,   # <-- NEW
) -> pd.DataFrame:
    if not pair_rows:
        return pd.DataFrame()

    node2cid, comps = _discover_components_with_ids(
        pair_rows,
        min_platforms=min_platforms,
        require_all_platforms=require_all_platforms,
        require_term_intersection=require_term_intersection,
    )
    if not comps:
        return pd.DataFrame()

    # Build quick edge index for score/terms aggregation
    edges_by_root: dict[int, list[tuple[Node, Node, ScoreDict, list[str]]]] = defaultdict(list)
    for pr in pair_rows:
        na, nb = (pr.plat_a, pr.topic_a), (pr.plat_b, pr.topic_b)
        cid_a = node2cid.get(na); cid_b = node2cid.get(nb)
        if cid_a is None or cid_b is None or cid_a != cid_b:
            continue
        edges_by_root[cid_a].append((na, nb, pr.scores, pr.common_terms))

    records: list[dict] = []
    for cid, nodes in enumerate(comps, start=1):
        # nodes belong to this component by construction
        plat_set = {p for (p, _) in nodes}
        platform_topics: Dict[str, List[int]] = defaultdict(list)
        for p, t in nodes:
            platform_topics[p].append(t)
        platform_topics = {p: sorted(set(ts)) for p, ts in platform_topics.items()}
        platform_topic_counts = {p: len(ts) for p, ts in platform_topics.items()}

        # scores/terms from contributing edges
        score_acc = defaultdict(list)
        term_sets = []
        # Rebuild edges for this component, but only those whose endpoints are in this comp
        comp_edges = []
        nodes_set = set(nodes)
        edge_terms_all = _aggregate_edge_terms(pair_rows)  # small overhead; cache globally if you want
        for pr in pair_rows:
            na, nb = (pr.plat_a, pr.topic_a), (pr.plat_b, pr.topic_b)
            if na in nodes_set and nb in nodes_set:
                if not require_term_intersection:
                    comp_edges.append((na, nb, pr.scores, pr.common_terms))
                else:
                    key = tuple(sorted([na, nb]))
                    if edge_terms_all.get(key, {}).get("intersection"):
                        comp_edges.append((na, nb, pr.scores, pr.common_terms))

        for _, _, sc, ct in comp_edges:
            for k, v in sc.items():
                score_acc[k].append(v)
            if ct:
                term_sets.append(set(ct))
        avg_scores = {f"avg_{k}": (sum(vs) / len(vs) if vs else math.nan) for k, vs in score_acc.items()}
        if term_sets:
            union_terms = sorted(set().union(*term_sets))
            inter_terms = sorted(set.intersection(*term_sets)) if len(term_sets) > 1 else sorted(next(iter(term_sets)))
        else:
            union_terms, inter_terms = [], []

        n_nodes = len(nodes)
        n_edges = len(comp_edges)
        density = (2 * n_edges) / (n_nodes * (n_nodes - 1)) if n_nodes > 1 else 0.0

        records.append({
            "cluster_id": cid,  # <— STABLE, matches graph
            "n_platforms": len(plat_set),
            "platforms": ";".join(sorted(plat_set)),
            "n_nodes": n_nodes,
            "n_edges": n_edges,
            "density": density,
            "platform_topic_counts": json.dumps(platform_topic_counts, ensure_ascii=False),
            "platform_topics_json": json.dumps(platform_topics, ensure_ascii=False),
            **avg_scores,
            "union_common_terms": ", ".join(union_terms[:common_terms_cap]),
            "intersection_common_terms": ", ".join(inter_terms[:common_terms_cap]),
        })

    out_df = pd.DataFrame(records).sort_values(["n_platforms", "n_edges"], ascending=[False, False])
    if save_path:
        os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
        out_df.to_csv(save_path, index=False, encoding="utf-8")
    return out_df


# -----------------------------
# Graph building on top of shared rows
# -----------------------------
def build_cross_platform_graph(
    pair_rows: List[PairRow],
    term_weight_metric: Optional[str] = None,
    min_term_freq: float = 1.0,
    max_terms_counted: int = 5,
    max_terms_label: int = 3,
    min_platforms_component: int = 3,
    require_all_platforms_component: bool = False,
    require_term_intersection: bool = False,   # <-- NEW
) -> "nx.Graph":
    if nx is None:
        raise ImportError("networkx not installed. pip install networkx pyvis")

    # Use the SAME component discovery as the CSV summary
    node2cid, comps = _discover_components_with_ids(
        pair_rows,
        min_platforms=min_platforms_component,
        require_all_platforms=require_all_platforms_component,
        require_term_intersection=require_term_intersection,
    )

    score_cols = ["cos_kw", "jacc", "contain", "ngram3"]
    edge_acc: Dict[EdgeKey, Dict] = {}
    node_meta: Dict[Node, Dict] = {}

    def _node(p: str, t: int) -> Node:
        return (p, t)

    edge_terms = _aggregate_edge_terms(pair_rows)
    def edge_has_intersection(na: Node, nb: Node) -> bool:
        key = tuple(sorted([na, nb]))
        et = edge_terms.get(key)
        return bool(et and et["intersection"])

    for pr in pair_rows:
        na, nb = (pr.plat_a, pr.topic_a), (pr.plat_b, pr.topic_b)
        if na not in node2cid or nb not in node2cid:
            continue
        if require_term_intersection and not edge_has_intersection(na, nb):
            continue

        if na not in node_meta:
            node_meta[na] = {"label": pr.label_a or f"{pr.plat_a}:{pr.topic_a}", "doc_count": pr.docs_a}
        if nb not in node_meta:
            node_meta[nb] = {"label": pr.label_b or f"{pr.plat_b}:{pr.topic_b}", "doc_count": pr.docs_b}

        ek = tuple(sorted([na, nb]))
        if ek not in edge_acc:
            edge_acc[ek] = {"files": set(), "score_sums": defaultdict(float), "score_counts": defaultdict(int), "terms_set": set()}
        rec = edge_acc[ek]
        if pr.source_file:
            rec["files"].add(pr.source_file)
        for sc, val in pr.scores.items():
            rec["score_sums"][sc] += float(val)
            rec["score_counts"][sc] += 1
        for t in pr.common_terms:
            rec["terms_set"].add(t)

    # Build graph
    G = nx.Graph()
    for (plat, tid), meta in node_meta.items():
        G.add_node((plat, tid), platform=plat, topic_id=tid, label=meta["label"], doc_count=meta["doc_count"])
        # Assign the SAME cluster_id here
        G.nodes[(plat, tid)]["cluster_id"] = int(node2cid[(plat, tid)])

    for (na, nb), rec in edge_acc.items():
        avg_scores = {k: (rec["score_sums"][k] / rec["score_counts"][k]) for k in rec["score_sums"] if rec["score_counts"][k] > 0}
        key = tuple(sorted([na, nb]))
        et = edge_terms.get(key, {"union": set(), "intersection": set()})
        G.add_edge(
            na, nb,
            files=set(rec["files"]),
            common_terms=set(rec["terms_set"]),
            common_terms_union=set(et["union"]),
            common_terms_intersection=set(et["intersection"]),
            **avg_scores
        )

    # No relabeling, no re-enumeration. cluster_id already matches CSV.
    # (Optional) set platforms per node for convenience
    for comp in comps:
        plats = {p for (p, _) in comp}
        for n in comp:
            if n in G:
                G.nodes[n]["cluster_platforms"] = plats

    # degree, edge_strength, and node_top_terms same as before...
    nx.set_node_attributes(G, dict(G.degree()), "degree")
    for _, _, d in G.edges(data=True):
        prim = [d.get(k) for k in ("cos_kw", "jacc", "contain") if d.get(k) is not None]
        d["edge_strength"] = sum(prim) / len(prim) if prim else 0.5

    weight_metric = term_weight_metric if term_weight_metric in ("cos_kw", "jacc", "contain", "ngram3", "edge_strength") else None
    node_term_scores: Dict[Node, Dict[str, float]] = {n: defaultdict(float) for n in G.nodes()}
    for u, v, d in G.edges(data=True):
        terms = d.get("common_terms") or set()
        w = float(d.get(weight_metric, 1.0)) if weight_metric else 1.0
        for t in terms:
            node_term_scores[u][t] += w
            node_term_scores[v][t] += w
    for n, counter in node_term_scores.items():
        if not counter:
            G.nodes[n]["node_top_terms_full"] = []
            G.nodes[n]["node_top_terms"] = []
            continue
        items = sorted(counter.items(), key=lambda kv: kv[1], reverse=True)
        full_terms = [t for t, _ in items[:max_terms_counted]]
        G.nodes[n]["node_top_terms_full"] = full_terms
        G.nodes[n]["node_top_terms"] = full_terms[:max_terms_label]

    return G

import re
from collections import Counter

_WORD_RE = re.compile(r"[A-Za-z][A-Za-z\-]+")

def fill_missing_node_terms(
    G,
    max_terms: int = 5,
    stop_terms: set[str] | None = None,
    prefer_existing: bool = True,
) -> None:
    """
    Ensure every node has node_top_terms/node_top_terms_full.
    Fallback order:
      1) keep existing node_top_terms(_full) if present (unless prefer_existing=False)
      2) derive from incident edges' common_terms
      3) derive from label tokens (simple regex), minus stop terms
    """
    stop = {s.lower() for s in (stop_terms or set())}
    for n in G.nodes():
        existing_full = G.nodes[n].get("node_top_terms_full") or []
        existing_short = G.nodes[n].get("node_top_terms") or []
        if prefer_existing and (existing_full or existing_short):
            # normalize + cap
            full = list(dict.fromkeys([t for t in existing_full if t]))  # de-dup preserve order
            short = full[:max_terms] if full else existing_short[:max_terms]
            G.nodes[n]["node_top_terms_full"] = full or short
            G.nodes[n]["node_top_terms"] = short
            continue

        # (2) collect from edges' common_terms
        term_counter = Counter()
        for _, v, d in G.edges(n, data=True):
            terms = d.get("common_terms") or []
            for t in terms:
                t = str(t).lower().strip()
                if not t or t in stop:
                    continue
                term_counter[t] += 1

        filled: list[str] = []
        if term_counter:
            filled = [t for t, _ in term_counter.most_common(max_terms)]

        # (3) fallback to label tokens if still empty
        if not filled:
            label = G.nodes[n].get("label") or ""
            tokens = [m.group(0).lower() for m in _WORD_RE.finditer(label)]
            tokens = [t for t in tokens if t not in stop]
            # keep order but de-dup
            seen = set()
            ordered = []
            for t in tokens:
                if t not in seen:
                    seen.add(t)
                    ordered.append(t)
            filled = ordered[:max_terms] if ordered else []

        G.nodes[n]["node_top_terms_full"] = filled
        G.nodes[n]["node_top_terms"] = filled[:max_terms]


def graph_to_dataframes(G: "nx.Graph") -> Tuple[pd.DataFrame, pd.DataFrame]:
    nodes_records = []
    for (plat, tid), attrs in G.nodes(data=True):
        nodes_records.append({
            "node_id": f"{plat}:{tid}",
            "platform": plat,
            "topic_id": tid,
            "cluster_id": attrs.get("cluster_id"),
            "degree": attrs.get("degree"),
            "doc_count": attrs.get("doc_count"),
            "label": attrs.get("label")
        })
    edges_records = []
    for (a, b, attrs) in G.edges(data=True):
        edges_records.append({
            "source": f"{a[0]}:{a[1]}",
            "target": f"{b[0]}:{b[1]}",
            "cos_kw": attrs.get("cos_kw"),
            "jacc": attrs.get("jacc"),
            "contain": attrs.get("contain"),
            "ngram3": attrs.get("ngram3"),
            "files": ";".join(sorted(attrs.get("files", []))) if isinstance(attrs.get("files"), set) else attrs.get("files"),
            "common_terms": ", ".join(sorted(attrs.get("common_terms", [])))
        })
    return pd.DataFrame(nodes_records), pd.DataFrame(edges_records)


# -----------------------------
# Visualization (PyVis + fallbacks)
# -----------------------------
def ensure_pyvis_deps():
    try:
        import jinja2  # noqa: F401
    except ImportError as ie:
        raise ImportError("PyVis requires jinja2. Install it via: pip install jinja2") from ie
    try:
        import pyvis  # noqa: F401
    except ImportError as ie:
        raise ImportError("pyvis not installed. pip install pyvis") from ie
    return True


def visualize_graph_pyvis(
    G: "nx.Graph",
    html_path: str,
    physics: bool = True,
    notebook: bool = False,
    color_map: Optional[dict] = None,
    max_nodes: int = 300,
    legend: bool = True,
    legend_max_terms: int = 10,
) -> str:
    if nx is None:
        raise ImportError("networkx not installed. pip install networkx pyvis")
    ensure_pyvis_deps()
    from pyvis.network import Network  # type: ignore

    # possibly subgraph
    if G.number_of_nodes() > max_nodes:
        top_nodes = sorted(G.degree, key=lambda x: x[1], reverse=True)[:max_nodes]
        keep = {n for n, _ in top_nodes}
        H = G.subgraph(keep).copy()
    else:
        H = G.copy()

    # cluster colors
    cluster_ids = sorted({H.nodes[n].get("cluster_id") for n in H.nodes() if H.nodes[n].get("cluster_id") is not None})
    if cluster_ids:
        palette = [
            "#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b","#e377c2","#7f7f7f","#bcbd22","#17becf",
            "#393b79","#637939","#8c6d31","#843c39","#7b4173","#3182bd","#e6550d","#31a354","#756bb1","#636363"
        ]
        cluster_color_map = {cid: palette[i % len(palette)] for i, cid in enumerate(cluster_ids)}
    else:
        cluster_color_map = {}

    # fallback platform colors
    plat_set = {p for (p, _) in H.nodes()}
    if color_map is None:
        palette_plat = ["#9edae5","#17becf","#c7c7c7","#dbdb8d","#ff9896"]
        color_map = {p: palette_plat[i % len(palette_plat)] for i, p in enumerate(sorted(plat_set))}

    net = Network(height="850px", width="100%", notebook=notebook, bgcolor="#ffffff", font_color="#222")
    net.force_atlas_2based() if physics else net.barnes_hut()

    # nodes
    for (plat, tid), a in H.nodes(data=True):
        deg = a.get("degree", 1)
        size = 10 + 2 * (deg ** 0.5)
        cluster = a.get("cluster_id")
        label_base = a.get("label") or f"{plat}:{tid}"
        terms = a.get("node_top_terms") or []
        term_str = ", ".join(terms)
        label = f"{label_base[:60]}\n{term_str}" if term_str else label_base[:60]
        title_parts = [f"Platform: {plat}", f"Topic: {tid}", f"Degree: {deg}"]
        if a.get("doc_count") is not None:
            title_parts.append(f"Docs: {a['doc_count']}")
        if cluster:
            title_parts.append(f"Cluster: {cluster}")
        if a.get("node_top_terms_full"):
            title_parts.append("Top terms: " + ", ".join(a["node_top_terms_full"]))
        color_val = cluster_color_map.get(cluster) if cluster else color_map.get(plat, "#999999")
        net.add_node(f"{plat}:{tid}", label=label, color=color_val, title="<br>".join(title_parts), size=size, group=cluster if cluster else plat)

    # edges
    for (a_node, b_node, attrs) in H.edges(data=True):
        strength = attrs.get("edge_strength") or attrs.get("cos_kw") or attrs.get("jacc") or attrs.get("contain") or 0.5
        width = 1 + 3 * float(strength)
        tooltip_parts = []
        for k in ["cos_kw", "jacc", "contain", "ngram3", "edge_strength"]:
            if attrs.get(k) is not None:
                tooltip_parts.append(f"{k}: {attrs[k]:.3f}")
        if attrs.get("common_terms"):
            ct_sample = ", ".join(list(attrs["common_terms"])[:8])
            tooltip_parts.append(f"terms: {ct_sample}")
        net.add_edge(f"{a_node[0]}:{a_node[1]}", f"{b_node[0]}:{b_node[1]}", value=strength, width=width, title="<br>".join(tooltip_parts))

    out_dir = os.path.dirname(html_path)
    if out_dir and not os.path.exists(out_dir):
        os.makedirs(out_dir, exist_ok=True)
    net.show(html_path)

    # legend
    if legend and cluster_ids:
        from collections import Counter
        cluster_term_counter = {}
        for cid in cluster_ids:
            c_terms = Counter()
            for n in H.nodes():
                if H.nodes[n].get("cluster_id") == cid:
                    for t in H.nodes[n].get("node_top_terms_full", []):
                        c_terms[t] += 1
            cluster_term_counter[cid] = [t for t, _ in c_terms.most_common(legend_max_terms)]
        legend_rows = []
        for cid in cluster_ids:
            color = cluster_color_map.get(cid, "#999999")
            terms = ", ".join(cluster_term_counter[cid])
            legend_rows.append(f"<tr><td style='padding:4px;background:{color};color:#fff;'>Cluster {cid}</td><td style='padding:4px;'>{terms}</td></tr>")
        legend_html = f"""<!DOCTYPE html><html><head><meta charset='utf-8'><title>Cluster Legend</title>
<style>body{{font-family:Arial, sans-serif;}} table{{border-collapse:collapse;}} td{{border:1px solid #ccc;}}</style>
</head><body>
<h3>Cluster Legend</h3>
<table>{''.join(legend_rows)}</table>
<p>Graph file: {os.path.basename(html_path)}</p>
</body></html>"""
        legend_path = os.path.join(out_dir or ".", os.path.splitext(os.path.basename(html_path))[0] + "_legend.html")
        with open(legend_path, "w", encoding="utf-8") as lf:
            lf.write(legend_html)

    return html_path


def visualize_graph_with_fallback(
    G: "nx.Graph",
    html_path: str,
    static_png_path: Optional[str] = None,
    physics: bool = True,
    max_nodes: int = 400
) -> Dict[str, str]:
    """Try PyVis first; fall back to static PNG and Plotly."""
    results: Dict[str, str] = {}
    import traceback
    # PyVis
    try:
        pv_path = visualize_graph_pyvis(G, html_path, physics=physics, max_nodes=max_nodes)
        results["pyvis_html"] = pv_path
        return results
    except Exception as e:
        print("PyVis failed, falling back. Reason:", type(e).__name__, e)
        traceback.print_exc(limit=1)

    # Matplotlib static
    try:
        import matplotlib.pyplot as plt
        H = G
        if G.number_of_nodes() > max_nodes:
            top_nodes = sorted(G.degree, key=lambda x: x[1], reverse=True)[:max_nodes]
            keep = {n for n, _ in top_nodes}
            H = G.subgraph(keep).copy()
        plt.figure(figsize=(12, 10))
        pos = nx.spring_layout(H, seed=42, k=0.35)
        # cluster coloring (fallback to platform)
        cluster_ids = sorted({H.nodes[n].get("cluster_id") for n in H.nodes() if H.nodes[n].get("cluster_id") is not None})
        platforms = [p for (p, _) in H.nodes()]
        if cluster_ids:
            palette = plt.cm.tab20.colors
            cluster_color_map = {cid: palette[i % len(palette)] for i, cid in enumerate(cluster_ids)}
            node_colors = [cluster_color_map.get(H.nodes[n].get("cluster_id"), (0.6, 0.6, 0.6)) for n in H.nodes()]
            unique_plats = sorted(set(platforms))
            color_map = {p: plt.cm.tab10(i % 10) for i, p in enumerate(unique_plats)}
        else:
            unique_plats = sorted(set(platforms))
            color_map = {p: plt.cm.tab10(i % 10) for i, p in enumerate(unique_plats)}
            node_colors = [color_map[p] for p in platforms]
        sizes = [80 + 10 * (H.degree(n) ** 0.5) for n in H.nodes()]
        nx.draw_networkx_edges(H, pos, alpha=0.25, width=0.6)
        nx.draw_networkx_nodes(H, pos, node_color=node_colors, node_size=sizes, linewidths=0.2, edgecolors="k")
        for n in H.nodes():
            terms = H.nodes[n].get("node_top_terms") or []
            if terms:
                plt.text(pos[n][0], pos[n][1], ", ".join(terms[:3]), fontsize=6, color="black")
        legend_patches = [plt.Line2D([0],[0], marker='o', color='w', label=p, markerfacecolor=color_map[p], markersize=8) for p in unique_plats]
        plt.legend(handles=legend_patches, title="Platform", fontsize=8)
        plt.title("Cross-Platform Topic Graph (static fallback)")
        plt.axis("off")
        if static_png_path:
            os.makedirs(os.path.dirname(static_png_path) or ".", exist_ok=True)
            plt.savefig(static_png_path, dpi=200, bbox_inches="tight")
            results["static_png"] = static_png_path
        plt.close()
    except Exception as e2:
        print("Matplotlib fallback failed:", e2)

    # Plotly interactive (enhanced: node hover terms + click-to-filter by cluster)
    try:
        import json
        import plotly.graph_objects as go
        H2 = G
        if G.number_of_nodes() > max_nodes:
            top_nodes = sorted(G.degree, key=lambda x: x[1], reverse=True)[:max_nodes]
            keep = {n for n,_ in top_nodes}
            H2 = G.subgraph(keep).copy()

        # ensure we have top terms for hover
        fill_missing_node_terms(
            H2,
            max_terms=5,
            stop_terms={"longevity","healthy","aging","healthy aging","biohacking","anti","anti-aging","research","study","video","news"}
        )

        # Layout
        pos = nx.spring_layout(H2, seed=13, k=0.4)

        # Cluster ids
        node_cluster = {n: (H2.nodes[n].get("cluster_id") or 0) for n in H2.nodes()}
        clusters_sorted = sorted({cid for cid in node_cluster.values() if cid is not None})

        # Edges per cluster (so we can toggle them cleanly)
        edges_by_cluster = {cid: {"x": [], "y": []} for cid in (clusters_sorted or [0])}
        for a, b in H2.edges():
            cid = node_cluster.get(a) or node_cluster.get(b) or 0
            x0, y0 = pos[a]; x1, y1 = pos[b]
            edges_by_cluster.setdefault(cid, {"x": [], "y": []})
            edges_by_cluster[cid]["x"] += [x0, x1, None]
            edges_by_cluster[cid]["y"] += [y0, y1, None]

        edge_traces = []
        for cid in (clusters_sorted or [0]):
            edge_traces.append(
                go.Scatter(
                    x=edges_by_cluster[cid]["x"],
                    y=edges_by_cluster[cid]["y"],
                    mode="lines",
                    line=dict(width=0.5),
                    hoverinfo="none",
                    name=f"Edges (Cluster {cid})",
                    showlegend=False,
                    visible=True,
                )
            )

        # Node trace (hover shows top terms; customdata carries cluster id)
        node_x, node_y, hover_text, customdata = [], [], [], []
        for (plat, tid), attrs in H2.nodes(data=True):
            x, y = pos[(plat, tid)]
            node_x.append(x); node_y.append(y)
            cid = attrs.get("cluster_id") or 0
            terms = attrs.get("node_top_terms") or []
            term_str = ", ".join(terms[:5])
            label = attrs.get("label") or f"{plat}:{tid}"
            hover_text.append(
                f"<b>{label}</b><br>"
                f"Platform: {plat}<br>"
                f"Topic: {tid}<br>"
                f"Cluster: {cid}<br>"
                f"Top terms: {term_str}"
            )
            customdata.append(cid)

        node_trace = go.Scatter(
            x=node_x, y=node_y,
            mode="markers",
            name="Topics",
            hoverinfo="text",
            text=hover_text,
            hovertemplate="%{text}<extra></extra>",
            customdata=customdata,  # cluster id per node
            marker=dict(size=9, line_width=0.5),
            showlegend=False,
        )

        fig = go.Figure(
            data=[*edge_traces, node_trace],
            layout=go.Layout(
                title="Cross-Platform Topic Graph (Plotly: click a node to isolate its cluster)",
                title_x=0.5,
                margin=dict(l=5, r=5, b=5, t=40),
            ),
        )

        # Reset button (restores all visibility/opacities)
        fig.update_layout(
            updatemenus=[
                dict(
                    type="buttons",
                    x=1.0, xanchor="right",
                    y=1.12, yanchor="top",
                    direction="right",
                    buttons=[
                        dict(
                            label="Reset view",
                            method="relayout",  # we’ll do actual reset in JS; this keeps a UI button visible
                            args=[{}],
                        )
                    ],
                )
            ]
        )

        # ---- Correct post_script: plain JS (NO <script> wrapper) ----
        post_js = f"""
          (function() {{
            var gd = document.getElementsByClassName('js-plotly-plot')[0];
            var edgeTraceCount = {len(edge_traces)};
            var nodeTraceIndex = {len(edge_traces)};  // last trace is nodes
            var totalNodePoints = {len(node_x)};
            var nodeClusterIds = {json.dumps(customdata)};             // per-node cluster ids
            var edgeClusterIds = {json.dumps(clusters_sorted or [0])}; // one id per edge trace in order

            // === Helpers ===
            function setNodeOpacityForCluster(clusterId) {{
              var opac = new Array(totalNodePoints).fill(0.12);
              for (var i = 0; i < totalNodePoints; i++) {{
                if (nodeClusterIds[i] === clusterId) opac[i] = 1.0;
              }}
              Plotly.restyle(gd, {{'marker.opacity': [opac]}}, [nodeTraceIndex]);
            }}

            function setEdgesVisibleForCluster(clusterId) {{
              for (var t = 0; t < edgeTraceCount; t++) {{
                var vis = (edgeClusterIds[t] === clusterId);
                Plotly.restyle(gd, {{'visible': [vis]}}, [t]);
              }}
              // keep node trace visible
              Plotly.restyle(gd, {{'visible': [true]}}, [nodeTraceIndex]);
            }}

            function resetAll() {{
              Plotly.restyle(gd, {{'marker.opacity': [null]}}, [nodeTraceIndex]); // default opacity
              for (var t = 0; t < edgeTraceCount; t++) {{
                Plotly.restyle(gd, {{'visible': [true]}}, [t]);
              }}
              Plotly.restyle(gd, {{'visible': [true]}}, [nodeTraceIndex]);
            }}

            function highlightClusterById(raw) {{
              if (raw === null || raw === undefined) return;
              var cid = parseInt(String(raw).trim(), 10);
              if (!Number.isFinite(cid)) {{
                alert('Please enter a numeric cluster id.');
                return;
              }}
              // if the id is not present, still apply node opacity (it will dim all)
              setNodeOpacityForCluster(cid);
              setEdgesVisibleForCluster(cid);
            }}

            // === Click to isolate a cluster (existing behavior) ===
            gd.on('plotly_click', function(evt) {{
              if (!evt || !evt.points || !evt.points.length) return;
              var p = evt.points[0];
              if (p.curveNumber !== nodeTraceIndex) return; // only react to node clicks
              var clusterId = p.customdata;
              setNodeOpacityForCluster(clusterId);
              setEdgesVisibleForCluster(clusterId);
            }});

            // Double-click anywhere to reset
            gd.on('plotly_doubleclick', function() {{ resetAll(); }});
            // Expose in console
            window.resetClusterView = resetAll;
            window.highlightCluster = highlightClusterById;

            // === Inject a small floating control panel ===
            var panel = document.createElement('div');
            panel.style.position = 'absolute';
            panel.style.top = '10px';
            panel.style.right = '10px';
            panel.style.zIndex = 9999;
            panel.style.background = 'rgba(255,255,255,0.9)';
            panel.style.border = '1px solid #ccc';
            panel.style.borderRadius = '8px';
            panel.style.padding = '8px 10px';
            panel.style.boxShadow = '0 2px 6px rgba(0,0,0,0.15)';
            panel.style.font = '12px/1.2 -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, Helvetica, Arial, sans-serif';

            var label = document.createElement('label');
            label.textContent = 'Cluster ID: ';
            label.style.marginRight = '6px';
            label.htmlFor = 'clusterInput';

            var input = document.createElement('input');
            input.type = 'text';
            input.id = 'clusterInput';
            input.size = 6;
            input.placeholder = 'e.g., 12';
            input.style.marginRight = '6px';

            var btnHi = document.createElement('button');
            btnHi.textContent = 'Highlight';
            btnHi.style.marginRight = '6px';
            btnHi.onclick = function() {{
              highlightClusterById(input.value);
            }};

            var btnReset = document.createElement('button');
            btnReset.textContent = 'Reset';
            btnReset.onclick = function() {{
              input.value = '';
              resetAll();
            }};

            panel.appendChild(label);
            panel.appendChild(input);
            panel.appendChild(btnHi);
            panel.appendChild(btnReset);

            // Insert panel into Plotly container
            // The Plotly root is the parent of gd (div with class 'js-plotly-plot')
            var root = gd.parentNode;
            root.style.position = 'relative';
            root.appendChild(panel);
          }})();
        """

        # Write HTML correctly: let Plotly handle adding the <script> tag
        plotly_html = html_path.replace(".html", "_plotly.html")
        os.makedirs(os.path.dirname(plotly_html) or ".", exist_ok=True)
        fig.write_html(
            plotly_html,
            include_plotlyjs="cdn",
            full_html=True,
            post_script=post_js,   # IMPORTANT: this must be plain JS, no <script> wrapper
        )
        results["plotly_html"] = plotly_html
        print(f"Plotly interactive HTML saved to {plotly_html}")
    except Exception as e3:
        print("Plotly fallback failed:", e3)

    return results

# -----------------------------
# IDE-friendly entrypoint (no argparse)
# -----------------------------
def run_pipeline(
    pair_files: list[str],
    outdir: str = "multi-platform/min_3_platforms",
    min_platforms: int = 3,
    require_all_platforms: bool = False,
    drop_low_scores: bool = False,
    min_cos: float = 0.0,
    min_jacc: float = 0.0,
    min_contain: float = 0.0,
    stop_terms: Optional[set[str]] = None,
    term_weight: Optional[str] = None,
    html_name: str = "topic_graph.html",
    static_png: Optional[str] = "topic_graph.png",
    max_nodes: int = 400,
    require_term_intersection: bool = False,   # <-- NEW
):
    """Run end-to-end pipeline from your IDE (PyCharm/VScode/Jupyter)."""
    os.makedirs(outdir, exist_ok=True)

    # 1) Load pairs (shared loader)
    pair_rows = load_pair_csvs(
        pair_files,
        drop_low_scores=drop_low_scores,
        min_cos_kw=min_cos,
        min_jacc=min_jacc,
        min_contain=min_contain,
        stop_terms=stop_terms or set(),
    )
    if not pair_rows:
        print("No rows loaded. Check paths/filters.")
        return {"clusters_path": None, "nodes_csv": None, "edges_csv": None, "viz": {}}

    # 2) Clusters
    clusters_path = os.path.join(outdir, "topic_clusters.csv")
    clusters = summarize_cross_platform_pairs(
        pair_rows,
        min_platforms=min_platforms,
        require_all_platforms=require_all_platforms,
        save_path=clusters_path,
        require_term_intersection=require_term_intersection,  # NEW
    )
    print(f"Clusters: {len(clusters)} saved -> {clusters_path}")

    # 3) Graph + tables
    if nx is None:
        print("networkx not installed; skipping graph.")
        return {"clusters_path": clusters_path, "nodes_csv": None, "edges_csv": None, "viz": {}}

    G = build_cross_platform_graph(
        pair_rows,
        term_weight_metric=term_weight,
        min_term_freq=1.0,
        max_terms_counted=5,
        max_terms_label=3,
        min_platforms_component=min_platforms,
        require_all_platforms_component=require_all_platforms,
        require_term_intersection=require_term_intersection,  # NEW
    )
    nodes_df, edges_df = graph_to_dataframes(G)
    nodes_csv = os.path.join(outdir, "topic_graph_nodes.csv")
    edges_csv = os.path.join(outdir, "topic_graph_edges.csv")
    nodes_df.to_csv(nodes_csv, index=False, encoding="utf-8")
    edges_df.to_csv(edges_csv, index=False, encoding="utf-8")
    print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges. Saved nodes/edges CSV.")

    # 4) Visualization (PyVis with fallbacks)
    html_path = os.path.join(outdir, html_name)
    png_path = os.path.join(outdir, static_png) if static_png else None
    viz_results = visualize_graph_with_fallback(
        G, html_path, static_png_path=png_path, physics=True, max_nodes=max_nodes
    )
    print("Visualization results:", viz_results)

    return {
        "clusters_path": clusters_path,
        "nodes_csv": nodes_csv,
        "edges_csv": edges_csv,
        "viz": viz_results,
        "graph": G,
        "clusters_df": clusters,
        "nodes_df": nodes_df,
        "edges_df": edges_df,
    }


# --- Example config block for running from IDE ---
if __name__ == "__main__":
    # Edit these lists/values in your IDE and run this file.
    PAIR_FILES = [
        "pubmed_simple-reddit/round_3/topic-pairs/keyword_pairs_2.csv",
        "pubmed_simple-youtube/round_3/topic-pairs/keyword_pairs_2.csv",
        "news-reddit/round_2/topic-pairs/keyword_pairs_2.csv",
        "news-youtube/round_2/topic-pairs/keyword_pairs_2.csv",
        "news-pubmed_simple/round_2/topic-pairs/keyword_pairs_2.csv",
        "reddit-youtube/round_4/topic-pairs/keyword_pairs_2.csv",
    ]

    RESULTS = run_pipeline(
        pair_files=PAIR_FILES,
        outdir="multi-platform/more_stricter_clustering",
        min_platforms=3,
        require_all_platforms=False,
        drop_low_scores=True,
        min_cos=0.0,
        min_jacc=0.0,
        min_contain=0.0,
        stop_terms={},
        term_weight="edge_strength",
        html_name="topic_graph.html",
        static_png="topic_graph.png",
        max_nodes=400,
        require_term_intersection=True,  # <--- turn it on
    )
    # You can inspect RESULTS['clusters_df'], RESULTS['nodes_df'], RESULTS['edges_df'] in the debugger.

---
<!-- nav-footer -->

← [01 cross platform matching](../../../../../SocialMedia/Reddit/notebooks/BERTopic/04_topic_matching/01_cross_platform_matching.ipynb) | [Project Overview](../../../../../PROJECT_OVERVIEW.ipynb) | [03 pair visuals](../../../../../SocialMedia/Reddit/notebooks/BERTopic/04_topic_matching/03_pair_visuals.ipynb) →
